# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a guided template for loading and exploring the [FAIR² dataset](https://doi.org/10.71728/senscience.y7m0-f273) using the `mlcroissant` library, referencing all entities by their Croissant `@id` fields for reproducibility and clear schema linkage.

### Dataset Source
The dataset source is defined by a Croissant schema at the following URL:

```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load the dataset metadata and the records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}\n")
print(f"Published: {metadata.date_published}")
print(f"License: {metadata.license}")

## 2. Data Overview
Let's review the available record sets in the dataset, along with fields and their Croissant `@id`s.

Here, each record set and field is referenced by its unique `@id`, which you should use for all further referencing and extraction.

In [ ]:
# List all available record sets and their fields, referencing @id fields
record_sets = list(dataset.record_sets)
print(f"Total record sets: {len(record_sets)}\n")

for rs in record_sets:
    print(f"Record set: {rs['@id']} | Name: {rs['name']}")
    if 'field' in rs:
        if isinstance(rs['field'], list):
            for f in rs['field']:
                if isinstance(f, dict):
                    field_id = f.get('@id', str(f))
                else:
                    field_id = f
                print(f"  Field: {field_id}")
        elif isinstance(rs['field'], dict):
            print(f"  Field: {rs['field'].get('@id', str(rs['field']))}")
    print()

# Store just the record set @ids for use below
record_set_ids = [rs['@id'] for rs in record_sets]

## 3. Data Extraction
We'll load the data from each record set into a Pandas DataFrame, referencing each record set and field by its `@id`.

Feel free to select only the record sets that are relevant for your analysis.

In [ ]:
# Extract data from each record set by its @id
dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading records for record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Columns: {df.columns.tolist()} | Rows: {len(df)}\n")
    else:
        print("No records found.\n")

# For demonstration, select the first available record set for inspection
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"Preview of record set {first_rs_id}:")
    display(dataframes[first_rs_id].head())
else:
    print("No dataframes loaded: check that the dataset exposes record sets with data.")

## 4. Exploratory Data Analysis (EDA)
Now, let's perform some basic data processing. We'll demonstrate this for a selected record set and fields.

- **Filtering** by a numeric field (referenced by its `@id`)
- **Normalizing** that numeric field
- **Grouping** by a categorical/group field (again using its `@id`)

You should update the variable values (`numeric_field_id`, `group_field_id`) with the exact `@id` values found in the overview if you want to focus your analysis.

In [ ]:
# Example: EDA for the first loaded record set & fields (replace ids as needed)
if dataframes:
    # Use the first available df, set the corresponding record set id
    record_set_id = first_rs_id
    df = dataframes[record_set_id]

    # Print field @ids for reference
    print("Available columns (field @ids):")
    print(df.columns.tolist())
    
    # --- Substitute these with real field @ids if needed ---
    numeric_field_id = None
    group_field_id = None
    # Try to guess some candidates based on column names
    for col in df.columns:
        if numeric_field_id is None and pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
        if group_field_id is None and pd.api.types.is_string_dtype(df[col]):
            group_field_id = col
    # If you know the exact @id, replace the variables here
    print(f"Selected numeric field: {numeric_field_id}")
    print(f"Selected group field: {group_field_id}\n")
    # Proceed with EDA on these fields
    if numeric_field_id is not None:
        # Filtering for values greater than a chosen threshold (10 for demo, adjust as needed)
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}: {len(filtered_df)}")
        display(filtered_df[[numeric_field_id]].head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by the group field (if present)
        if group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
else:
    print("No suitable data found for EDA. Please check your dataset record sets and fields.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field (referenced by its `@id`), and if possible, show its grouping by a key attribute.

Adjust the plotting code if you want to analyze a specific record set or field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id], kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouping is feasible
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=40, ha='right')
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load, explore, and analyze a FAIR² dataset using the `mlcroissant` library, always referencing Croissant entities via their `@id` fields for traceability and reproducibility.

- **Metadata**: Loaded and summarized key dataset information.
- **Record sets & fields**: Catalogued all available record sets and their field `@id`s.
- **Data extraction**: Loaded tabular data for each available record set, accessed using their `@id`.
- **EDA/Visualization**: Performed introductory analysis and visualization by referencing data schema via `@id` fields.

This workflow allows you to build portable, repeatable analyses for any dataset using the Croissant/FAIR² ecosystem.